# Align Sample


Please note 
1. this helper notebook was created with slurm cluster in mind please modify according to your own cluster
2. We expect that the `patientID` and the `sample type` is joined using an underscore `_` and themselves dont contain these modify this logic accordingly in the path determination


Please copy this notebook on a per project basis

In [4]:
import glob
import os
import pandas as pd

In [5]:
# Lines to be modified 

ALIGNMENT_PIPELINE_PATH = "path/to/cloned/directory"
TARGET_FILE = "Agilent_SureSelectv5.target.bed"
BAIT_FILE = "Agilent_SureSelectv5.bait.bed"
METADATA_PATH = "/path/to/metadata_file"



BASE_OUTPUT_PATH= "/path/to/output/directory" 
SCRIPT_PATH= "/path/to/scripts/directory" 

ADAPTER_FORWARD_READ="adapter_forward_read"
ADAPTER_REVERSE_READ="adapter_reverse_read"

# Agilent v8 and above can be provided with MBC which needs to be removed or be run using a umi aware alignment pipeline
# Please modifiy this accordingly
ADDITIONAL_CUTADPT_PARAMETER="-u 5"

SEQ_TYPE="WES"

CONDA_ACTIVATE_PATH="~/miniforge3/activate"
SLURM_SMK_PROFILE="~/smk-profile-v0.1"
WORKFLOW_PROFILE="profile/" # please edit the config.yaml inside the path

In [ ]:
fastq_files=glob.glob(f'{FASTQ_FILE_PATH}/*fastq', recursive=True)

In [ ]:
df=pd.read_csv(METADATA_PATH)

In [ ]:
import yaml
from pathlib import Path

workdir = Path(
    f"{BASE_OUTPUT_PATH}/view-by-pid/"
)


bash_folder = Path(f"{SCRIPT_PATH}")

os.makedirs(bash_folder, exist_ok=True)
os.makedirs(workdir, exist_ok=True)


yaml_dict = yaml.safe_load(
    Path(
        "config/config.yaml"
    ).read_text()
)

count = 0
for i in df.SAMPLE_NAME.unique():
    count += 1
    
    yaml_dict["work_dir"] = (
        workdir / i.split("_")[0] / i.split("_")[1] / "merged-alignment"
    )
    
    yaml_dict["log_dir"] = yaml_dict["work_dir"] / "logs"
    os.system("mkdir -p " + str(yaml_dict["work_dir"]))
    yaml_dict["pid"] = i.split("_")[0]
    yaml_dict["sample"] = i.split("_")[1]
    config_file = yaml_dict["work_dir"] / (i + ".yaml")
    yaml_dict["work_dir"] = str(yaml_dict["work_dir"])
    yaml_dict["log_dir"] = str(yaml_dict["log_dir"])
    
    yaml_dict["target_regions"] = TARGET_FILE
    yaml_dict["bait_regions"] = BAIT_FILE
    
    yaml_dict["SeqType"] = SEQ_TYPE
    yaml_dict["library_prep_kit"] = df[df.SAMPLE_NAME==i, "LIBRARY_PREP_KIT"].unique()[0]

    if len(df[df.SAMPLE_NAME==i, "LIBRARY_PREP_KIT"].unique())>1:
        raise ValueError("Same Sample cannot have different library prep kit")
        
    yaml_dict["trim_adapters"] = True   
    yaml_dict['adapter_forward_read'] = ADAPTER_FORWARD_READ
    yaml_dict['adapter_reverse_read']= ADAPTER_REVERSE_READ
    yaml_dict['extra_cutadapt_params']=ADDITIONAL_CUTADPT_PARAMETER
    if 'extra_cutadapt_params' in yaml_dict and ADDITIONAL_CUTADPT_PARAMETER =="":
        del yaml_dict['extra_cutadapt_params']

    yaml_dict["metadata"] = METADATA_PATH
    
    with open(config_file, "w") as ff:
        yaml.dump(yaml_dict, ff)
    """
    #####

    Please modifiy according to your cluster
    ####
    """
    cmd = (
        "module purge;source"
        f"{CONDA_ACTIVATE_PATH}"
        " alignment-base && cd "
        + yaml_dict["work_dir"]
        + " "
        + f"&& snakemake --profile {SLURM_SMK_PROFILE} "
        + f"--workflow-profile {WORKFLOW_PROFILE} "
        + " --use-singularity --use-conda --conda-frontend mamba -j 10 "
        + f"--configfile {str(config_file)} "
        + f"--snakefile {ALIGNMENT_PIPELINE_PATH}/workflow/Snakefile"
    )
    with open(bash_folder / (str(count) + ".sh"), "w") as handle:
        handle.write(cmd)

In [ ]:
count

In [ ]:
df.shape